# Visual Computing: Object Tracking and Motion Analysis
 In this coursework, you will implement various object tracking algorithms and motion analysis techniques using computer vision. The goal is to understand and apply different tracking approaches, analyse their performance, and evaluate their effectiveness under real-world conditions. This coursework is worth 7% of the total marks for the unit.

### Change Detection Using GMM [10%]
#### Objective:
Implement Change Detection using a Gaussian Mixture Model (GMM) for object tracking from scratch. Follow the algorithm outlined in the provided lecture slide for modelling pixel changes.
cv2.createBackgroundSubtractorMOG2, is not allowed.
#### Implementation Details:
Model the image as a mixture of Gaussians and classify each pixel as foreground or background. This will be done based on pixel probability distributions, as explained in the lecture. Update the Gaussian distributions per pixel to decide whether a pixel belongs to the background or foreground. Evaluate the effectiveness of the model in classifying moving objects. You can use cv2.VideoCapture(), cv2.cvtColor(),cv2.COLOR_BGR2GRAY(), cv2.waitKey().


In [ ]:
# Gaussian Mixture Model (GMM) for Change Detection
# -------------------------------------------------
# Instructions: Below, I provide a code structure as a guidance to get you started. Of course you can use your own code structure to achieve the objective. Your marks will NOT be deducted if you use your own code structure :) 
# Complete the sections marked with '### ENTER YOUR CODE HERE ###' to implement 
# the Gaussian Mixture Model for background subtraction and change detection.

import cv2
import numpy as np

# Initialize the GMMBackgroundSubtractor Class
class GMMBackgroundSubtractor:
    def __init__(self, frame_shape, num_gaussians= """ENTER NUMBER OF GAUSSIANS""", learning_rate="""ENTER LEARNING RATE""", threshold="""ENTER THRESHOLD"""):
        """
        Initialize Gaussian parameters: means, variances, and weights.

        Args:
            frame_shape (tuple): Shape of the input frame (height, width).
            num_gaussians (int): Number of Gaussian models per pixel.
            learning_rate (float): Rate at which the model updates.
            threshold (float): Threshold for matching a pixel to a Gaussian.
        """
        self.num_gaussians = num_gaussians
        self.learning_rate = learning_rate
        self.threshold = threshold
        
        # Initialize the means, variances, and weights for each Gaussian
        ### ENTER YOUR CODE HERE ###
       
        

    def apply(self, frame):
        """
        Apply GMM to detect foreground objects.

        Args:
            frame (ndarray): Input video frame.

        Returns:
            foreground_mask (ndarray): Binary mask indicating foreground pixels.
        """
        # Convert frame to grayscale if it isn't already
        if len(frame.shape) == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        frame = frame.astype(np.float32)
        
        # Calculate the absolute difference between the frame and Gaussian means
        ### ENTER YOUR CODE HERE ###

        # Check which pixels match any of the Gaussians
        ### ENTER YOUR CODE HERE ###

        ### ENTER YOUR CODE HERE ###
        # Step 1: Update matched Gaussians (means, variances, weights)
        
        # Normalize weights so they sum to 1
        ### ENTER YOUR CODE HERE ###

        ### ENTER YOUR CODE HERE ###
        # Step 2: Classify Foreground
      
        
        return foreground_mask.astype(np.uint8) * 255

# ------------------- Main Code to Run the GMM ------------------- #

# Load Video
video_path = 'input_video.mp4'  # Replace with your video file
cap = cv2.VideoCapture(video_path)


# Read the first frame to get the frame shape


# Initialize GMM Subtractor
gmm_subtractor = GMMBackgroundSubtractor(frame.shape[:2])

# Process the video frame by frame
### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###

# Release resources and close windows
cap.release()
cv2.destroyAllWindows()


### Custom Lucas-Kanade (with OpenCV GMM) [10%]
#### Objective
In this task, students will be using Lucas-Kanade Optical Flow to track detected moving objects. You can use the OpenCV GMM cv2.createBackgroundSubtractorMOG2() for this section.  
#### Implementation Details:
Detect foreground objects using the OpenCV GMM, then extract feature points inside the foreground mask. Thereafter, implement and apply your custom Lucas-Kanade Optical Flow to track moving objects frame-by-frame. Visualise this via Motion vectors which are optical flow arrows showing direction and magnitude of object movement. You can use cv2.Sobel(), cv2.VideoCapture(), cv2.goodFeaturesToTrack(), cv2.cvtColor(),cv2.COLOR_BGR2GRAY(), cv2.waitKey(), and for visualization cv2.circle(),cv2.arrowedLine. 


In [ ]:
import cv2
import numpy as np

class LucasKanadeTracker:
    def __init__(self):
        # Parameters for Good Features to Track
        self.feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)
        
        # Parameters for Lucas-Kanade Optical Flow
        self.lk_params = dict(winSize=(15, 15), 
                              maxLevel=2, 
                              criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

    def detect_features(self, frame, mask):
        """
        Detects corner features (keypoints) in the current frame,
        but only within the foreground mask (i.e., moving objects).
        """
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return cv2.goodFeaturesToTrack(gray, mask=mask, **self.feature_params)

    def calculate_optical_flow(self, prev_frame, curr_frame, prev_points):
        """
        Tracks where the previously detected points have moved
        in the new frame using Lucas-Kanade Optical Flow.
        """
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        
        # Estimate the new positions using LK optical flow
        new_points, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_points, None, **self.lk_params)
        return new_points, status

    def draw_motion_vectors(self, frame, prev_points, new_points, status):
        """
        Draws motion vectors (arrows) showing where each feature moved.
        - Green arrow: movement direction from previous point to new point
        - Red circle: new point location
        """
        for i, (new, old) in enumerate(zip(new_points, prev_points)):
            if status[i]:  # Only process valid points
                a, b = map(int, new.ravel())
                c, d = map(int, old.ravel())
                cv2.arrowedLine(frame, (c, d), (a, b), (0, 255, 0), 2, tipLength=0.4)
                cv2.circle(frame, (a, b), 3, (0, 0, 255), -1)
        return frame


# Load Video
cap = cv2.VideoCapture('Lucas-Kanade-input_video.mp4') # Replace with your video file


### CODE FOR VIDEO PROCESSING ###

# Initialize GMM Background Subtractor to help isolate moving objects
bg_subtractor = cv2.createBackgroundSubtractorMOG2()

# Read the first frame from the input video
ret, prev_frame = cap.read()
if not ret:
    print("Error: Couldn't read video.")
    cap.release()
    exit()

# Initialize Lucas-Kanade Tracker
lk_tracker = LucasKanadeTracker()

# Apply background subtraction to find foreground in the first frame
foreground_mask = bg_subtractor.apply(prev_frame)
prev_points = lk_tracker.detect_features(prev_frame, foreground_mask)

# Process the video frame-by-frame
while cap.isOpened():
    ret, curr_frame = cap.read()
    if not ret:
        break

    # Apply background subtraction
    foreground_mask = bg_subtractor.apply(curr_frame)

    # Only track if we have points from the last frame
    if prev_points is not None and len(prev_points) > 0:
        # Compute optical flow
        new_points, status = lk_tracker.calculate_optical_flow(prev_frame, curr_frame, prev_points)

        # Draw the motion arrows and updated point locations
        tracked_frame = lk_tracker.draw_motion_vectors(curr_frame.copy(), prev_points, new_points, status)

        # Show results
        cv2.imshow('Foreground Mask', foreground_mask)
        cv2.imshow('Optical Flow Tracking', tracked_frame)

        # Update for next frame
        prev_frame = curr_frame.copy()
        prev_points = new_points[status == 1].reshape(-1, 1, 2) 

    # Press 'q' to exit
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


### Template Matching for Object Tracking [10%] 
#### Objective:
Students will implement template matching using a weighted histogram matching technique from scratch to track a target object across multiple frames in a video. Experiment with NCC to obtain the results.
#### Implementation Details:
User manually selects a target object in the first frame. Apply template matching to locate the object in subsequent frames. Experiment with the similarity metric:  NCC (Normalized Cross-Correlation) between template & search region. Draw a bounding box around the detected object in each frame (The best match is marked with a rectangle in each frame). You can use cv2.matchTemplate(),cv2.selectROI(),cv2.minMaxLoc (),cv2.TM_CCORR_NORMED (),cv2.COLOR_BGR2GRAY (), cv2.VideoCapture(), cv2.cvtColor(), cv2.waitKey(),cv2.normalize().



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def compute_epanechnikov_kernel():
    r = 8
    x = np.arange(-7.5, 8.5)
    y = np.arange(-7.5, 8.5)

    X, Y = np.meshgrid(x, y)

    distances = X**2 + Y**2

    kernel = np.maximum(0, 1 - (distances / r**2))

    return kernel

ep_kernel = compute_epanechnikov_kernel()

class TemplateMatcher:
    def __init__(self):
        pass

    def select_template(self, frame):
        """
        Allow the user to manually select the target object in the first frame.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.selectROI() to select the region of interest (ROI).

        r = cv2.selectROI("select", frame)

        return frame[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]

    def match_template(self, frame, template, temp_hist):
        """
        Apply template matching using Normalized Cross-Correlation (NCC).
        """
        ### ENTER YOUR CODE HERE ###

        fr_width = frame.shape[1]
        fr_height = frame.shape[0]
        temp_width = template.shape[1]
        temp_height = template.shape[0]

        results = np.zeros((frame.shape[0] - template.shape[0], frame.shape[1] - template.shape[1]))
        
        for i in range(frame.shape[0] - template.shape[0]):
            for j in range(frame.shape[1] - template.shape[1]):
                histogram = np.zeros([256])
                for x in range(template.shape[0]):
                    for y in range(template.shape[1]):
                        value = int(frame[x+i, y+j])
                        histogram[value] += 1

                histogram = histogram.reshape(16, 16) / 255
                histogram = np.matmul(histogram, ep_kernel).astype(np.float32)

                result = cv2.matchTemplate(histogram, temp_hist, cv2.TM_CCORR_NORMED)
                results[i, j] = result[0][0]

        minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(results, None)

        return maxLoc

    def draw_bounding_box(self, frame, top_left, template_size):
        """
        Draw a bounding box around the detected object.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.rectangle() to draw the box.

        return cv2.rectangle(frame, top_left, (top_left[0] + template_size[1], top_left[1] + template_size[0]), (0,0,0), 2)

# Load Video
cap = cv2.VideoCapture('believecropevenlower.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###
success = 1
count = 0

tm = TemplateMatcher()

success, frame = cap.read()

template = tm.select_template(frame)

temp_grey = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

temp_width = template.shape[1]
temp_height = template.shape[0]

temp_hist = np.zeros([256])
for x in range(temp_grey.shape[0]):
    for y in range(temp_grey.shape[1]):
        value = int(temp_grey[x, y])
        temp_hist[value] += 1

temp_hist = temp_hist.reshape(16, 16) / 255
temp_hist = np.matmul(temp_hist, ep_kernel).astype(np.float32)

while success:
    success, img = cap.read()

    img_grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    ##plt.plot(temp_hist)
    #plt.show()

    topleft = tm.match_template(img_grey, temp_grey, temp_hist)

    img = tm.draw_bounding_box(img, topleft, (template.shape[0], template.shape[1]))

    cv2.imwrite("frames/f%d.jpg" % count, img)
    count += 1

cap.release()
cv2.destroyAllWindows()


(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)
(114, 147)


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


: 

### Improving Template Matching for Object Tracking [5% + 10%]
#### Objective:
After implementing template matching, students should explore potential improvements.  The key challenges with template matching are: 1. Scale changes 2. Rotation Changes 3. Brightness/ contrast changes and Occlusions.
#### Implementation Details:
[A] Multi-Scale Template Matching (Handling Scale Changes)
Instead of using a fixed-size template, track multiple scales: Resize the template to 3 different scales (of your choice) before matching. Match at different pyramid levels using cv2.pyrDown() and cv2.pyrUp(). You can use prebuilt functions: cv2.matchTemplate(),cv2.minMaxLoc(),cv2.resize().

[B] Rotation-Invariant Matching
Rotate the template at 3 different angles (of your choice)  and match each rotated version. You can use cv2.getRotationMatrix2D(),cv2.warpAffine().

[C] Using Feature-Based Matching Instead of Pixel-Based Matching
Template matching relies on raw pixel values, making it sensitive to changes. Instead of comparing pixel intensities, extract SIFT features and follow the algorithm for tracking by feature detection discussed in the lecture. You can use cv2.SIFT_create(), sift.detectAndCompute,cv2.BFMatcher(), cv2.drawMatches() function here to compare. 


In [ ]:
import cv2
import numpy as np

class ImprovedTemplateMatcher:
    def __init__(self):
        pass

    def multi_scale_matching(self, frame, template):
        """
        Perform multi-scale template matching to handle scale changes.
        """
        ### ENTER YOUR CODE HERE ###
        # Resize the template to different scales 
        pass

    def rotation_invariant_matching(self, frame, template):
        """
        Perform rotation-invariant template matching.
        """
        ### ENTER YOUR CODE HERE ###
        # Rotate the template at different angles 
        pass

    def feature_based_matching(self, frame, template):
        """
        Use SIFT feature detection for robust template matching.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.SIFT_create(), detectAndCompute() to extract features.
        # Match features using cv2.BFMatcher() and visualize with cv2.drawMatches().
        pass

    def draw_bounding_box(self, frame, top_left, template_size):
        """
        Draw bounding box around the detected object.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.rectangle() to draw the bounding box.
        pass

# Load Video
cap = cv2.VideoCapture('input_video.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###


cap.release()
cv2.destroyAllWindows()
